In [ ]:
# save_perm_pop_expand_pipeline.py
# Python replication of MATLAB savePermPop_expand.m for ONE ROI (vreg) at a time.
# Produces ONE NPZ per (subject, vreg), matching the MATLAB organization and logic.
# - Uses Spearman similarity between per-session RDM vectors by default
# - Builds image-wise session×session correlations and averages across images
# - Collapses matrices by session distance (off-diagonals) with mean
# - Deterministic permutations (rng=1), with optional "fixedFirst"
# - Batch runner auto-discovers subjects that have both required inputs and skips existing outputs

import os
import re
import numpy as np
from scipy.linalg import toeplitz
from scipy.spatial.distance import pdist
from google.colab import drive

# =========================
# CONFIG (edit if needed)
# =========================
BASE_FOLDER = "/content/drive/My Drive/V1_Drift/repDrift_expand"

# Visual region index (MATLAB: V1=1, V2=2, V3=3, hV4=4)
DEFAULT_VREG = 1

# Z-scoring option:
# 0='', 1='_zscore', 2='_zeroMean', 3='_equalStd', 4='_zeroROImean'
DEFAULT_TO_ZSCORE = 0

# Default between-session similarity metric
DEFAULT_METRIC = "rmse"  # "spearman" | "pearson" | "rmse" | "euclidean"

# =========================
# Mount Drive (idempotent)
# =========================
drive.mount('/content/drive', force_remount=False)

# =========================
# Helpers
# =========================
def _zstr(to_zscore: int) -> str:
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][to_zscore]

def _nanmean(x, axis=None):
    return np.nanmean(x, axis=axis)

def _subject_root(base_folder: str, vreg: int, isub: int) -> str:
    return os.path.join(base_folder, f"v{vreg}", f"Subject {isub}")

def _comb_npz_path(base_folder: str, vreg: int, isub: int, to_zscore: int) -> str:
    # Matches other scripts: regressSessCombineROI_sub{isub}{zscore}.npz
    return os.path.join(_subject_root(base_folder, vreg, isub),
                        f"regressSessCombineROI_sub{isub}{_zstr(to_zscore)}.npz")

def _sim_npz_path(base_folder: str, vreg: int, isub: int, to_zscore: int) -> str:
    return os.path.join(_subject_root(base_folder, vreg, isub),
                        f"simPopResp_v{vreg}_sub{isub}{_zstr(to_zscore)}.npz")

def _out_npz_path(base_folder: str, vreg: int, isub: int,
                  to_zscore: int, nperms: int, r2thresh: float, fixedFirst: bool, metric: str) -> str:
    fixedFirstStr = '_fixedFirst_' if fixedFirst else ''
    r2threshStr = ('' if r2thresh <= 0 else f"r2thresh{r2thresh:0.2f}".rstrip('0').rstrip('.'))
    name = f"permPop{fixedFirstStr}N{nperms}_v{vreg}_sub{isub}{_zstr(to_zscore)}{r2threshStr}_{metric}.npz"
    return os.path.join(_subject_root(base_folder, vreg, isub), name)

def _discover_subjects_with_inputs(base_folder: str, vreg: int, to_zscore: int) -> list[int]:
    """
    Return subject IDs that have BOTH:
      - regressSessCombineROI_sub{isub}{z}.npz
      - simPopResp_v{vreg}_sub{isub}{z}.npz
    """
    vfolder = os.path.join(base_folder, f"v{vreg}")
    if not os.path.isdir(vfolder):
        return []
    z = _zstr(to_zscore)
    subs = []
    for name in sorted(os.listdir(vfolder)):
        m = re.match(r"Subject\s+(\d+)$", name.strip())
        if not m:
            continue
        isub = int(m.group(1))
        root = os.path.join(vfolder, name)
        comb_ok = os.path.exists(os.path.join(root, f"regressSessCombineROI_sub{isub}{z}.npz"))
        sim_ok  = os.path.exists(os.path.join(root, f"simPopResp_v{vreg}_sub{isub}{z}.npz"))
        if comb_ok and sim_ok:
            subs.append(isub)
    return subs

# ---- masked similarity helpers ----
def _pearson_masked(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    x = x[mask]; y = y[mask]
    x_m = x - x.mean(); y_m = y - y.mean()
    denom = np.linalg.norm(x_m) * np.linalg.norm(y_m)
    if denom == 0:
        return np.nan
    return float((x_m @ y_m) / denom)

def _spearman_masked(x, y):
    from scipy.stats import rankdata
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return np.nan
    xr = rankdata(x[mask], method='average')
    yr = rankdata(y[mask], method='average')
    return _pearson_masked(xr, yr)

def _rmse_masked(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() == 0:
        return np.nan
    diff = x[mask] - y[mask]
    return float(np.sqrt(np.mean(diff**2)))

def _euclidean_masked(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() == 0:
        return np.nan
    diff = x[mask] - y[mask]
    return float(np.linalg.norm(diff))

def between_session_similarity(sessVec: np.ndarray, metric: str = "spearman") -> np.ndarray:
    """
    Pairwise similarity between session RDM vectors.
    sessVec: [nsessions, nvec]  (each row = flattened RDM for a session)
    metric: "spearman" | "pearson" | "rmse" | "euclidean"
    Returns: [nsessions, nsessions] similarity matrix.
    For distance metrics, returns NEGATIVE distances so that higher = more similar.
    """
    ns = sessVec.shape[0]
    R = np.full((ns, ns), np.nan, dtype=float)
    for i in range(ns):
        R[i, i] = 1.0
        xi = np.asarray(sessVec[i], float)
        for j in range(i+1, ns):
            xj = np.asarray(sessVec[j], float)
            if metric == "spearman":
                val = _spearman_masked(xi, xj)
            elif metric == "pearson":
                val = _pearson_masked(xi, xj)
            elif metric == "rmse":
                val = -_rmse_masked(xi, xj)  # negative so higher is better
            elif metric == "euclidean":
                val = -_euclidean_masked(xi, xj)
            else:
                raise ValueError(f"Unknown metric '{metric}'")
            R[i, j] = R[j, i] = val
    return R

# =========================
# Core (single subject, single vreg)
# =========================
def save_perm_pop_expand_npz(
    to_zscore: int = DEFAULT_TO_ZSCORE,
    nperms: int = 1000,
    r2thresh: float = 0.0,
    fixedFirst: bool = False,
    isub: int = 1,
    vreg: int = DEFAULT_VREG,
    base_folder: str = BASE_FOLDER,
    metric: str = DEFAULT_METRIC
) -> str:
    """
    Python replication of savePermPop_expand.m for ONE subject and ONE ROI (vreg),
    with selectable between-session similarity metric.

    Inputs used:
      - regressSessCombineROI_sub{isub}{z}.npz  (for nsplits and allRoiPrf)
      - simPopResp_v{vreg}_sub{isub}{z}.npz    (for voxSessResp, voxSessRespOri)
    Output:
      - permPop[_fixedFirst_]N{nperms}_v{vreg}_sub{isub}{z}[_r2threshX]_{metric}.npz
    """
    ROOT = _subject_root(base_folder, vreg, isub)
    os.makedirs(ROOT, exist_ok=True)

    zscoreStr = _zstr(to_zscore)
    distTypeRDM = 'correlation'  # for pdist() when building each session's RDM

    # --- Load combined-ROI regression meta for this subject (nsplits, allRoiPrf)
    comb_path = _comb_npz_path(base_folder, vreg, isub, to_zscore)
    if not os.path.exists(comb_path):
        raise FileNotFoundError(comb_path)
    C = np.load(comb_path, allow_pickle=True)
    if 'nsplits' not in C or 'allRoiPrf' not in C:
        raise KeyError(f"Missing 'nsplits' or 'allRoiPrf' in {comb_path}")
    nsplits = int(np.array(C['nsplits']).item())   # includes the mean split
    ns_no_mean = nsplits - 1                        # MATLAB uses nsessions = nsplits-1
    allRoiPrf = C['allRoiPrf'].item()              # dict: {1:{'r2':...}, 2:..., ...}

    # --- Load population simulations
    sim_path = _sim_npz_path(base_folder, vreg, isub, to_zscore)
    if not os.path.exists(sim_path):
        raise FileNotFoundError(sim_path)
    S = np.load(sim_path, allow_pickle=True)
    for key in ['voxSessResp', 'voxSessRespOri', 'simImgs', 'nsessions']:
        if key not in S:
            raise KeyError(f"Missing '{key}' in {sim_path}")
    voxSessResp = S['voxSessResp']        # [nvox, nsess_sim, nimg]
    voxSessRespOri = S['voxSessRespOri']  # [nvox, nsess_sim, nimg]
    simImgs = S['simImgs']
    nimg = int(simImgs.shape[0])

    # --- Use only the first (ns_no_mean) sessions
    Xfull = voxSessResp[:, :ns_no_mean, :]      # (vox, ns, nimg)
    Xofull = voxSessRespOri[:, :ns_no_mean, :]
    ns = Xfull.shape[1]
    minSessions = ns
    subSessions = np.ones(minSessions, dtype=int)

    # --- Voxel filter by pRF r2
    if vreg not in allRoiPrf or 'r2' not in allRoiPrf[vreg]:
        raise KeyError(f"ROI {vreg} missing 'r2' in allRoiPrf in {comb_path}")
    r2vec = np.array(allRoiPrf[vreg]['r2']).reshape(-1)
    good = np.isfinite(r2vec) & (r2vec > r2thresh)
    nvox_good = int(np.sum(good))
    if nvox_good == 0:
        raise RuntimeError(f"No voxels pass r2thresh={r2thresh} (sub {isub}, v{vreg})")

    # Slice to good voxels
    X  = Xfull[good, :, :].astype(float)   # (vox, ns, nimg)
    Xo = Xofull[good, :, :].astype(float)

    # --- Per-image session×session correlations (will average across images)
    imgCorr    = np.full((nimg, ns, ns), np.nan, dtype=float)
    imgCorrOri = np.full((nimg, ns, ns), np.nan, dtype=float)

    # --- Per-session RDM vectors (images×images)
    nvec = nimg * (nimg - 1) // 2
    sessVec    = np.full((ns, nvec), np.nan, dtype=float)
    sessVecOri = np.full((ns, nvec), np.nan, dtype=float)

    # Build per-image session×session corr (corr across vox, within image)
    for ii in range(nimg):
        M  = X[:,  :, ii].T   # (ns, vox)
        Mo = Xo[:, :, ii].T
        imgCorr[ii]    = np.corrcoef(M,  rowvar=True)
        imgCorrOri[ii] = np.corrcoef(Mo, rowvar=True)

    # Build per-session image×image RDMs + vectorize (distance across images)
    for ss in range(ns):
        Ms  = X[:,  ss, :].T  # (nimg, vox)
        Mso = Xo[:, ss, :].T
        sessVec[ss]    = pdist(Ms,  metric=distTypeRDM)
        sessVecOri[ss] = pdist(Mso, metric=distTypeRDM)

    # --- Between-session similarity using chosen metric (Spearman matches MATLAB default)
    R  = between_session_similarity(sessVec,    metric=metric)  # (ns, ns)
    Ro = between_session_similarity(sessVecOri, metric=metric)  # (ns, ns)

    # --- Average across images → session×session matrices
    avgImgCorrMat    = _nanmean(imgCorr[:, :minSessions, :minSessions], axis=0)       # (ns, ns)
    avgImgCorrMatOri = _nanmean(imgCorrOri[:, :minSessions, :minSessions], axis=0)

    # --- Distance matrix over sessions
    distMatrix = toeplitz(np.arange(minSessions))

    # --- Collapse by distance (mean along off-diagonals)
    betweenSessImg     = np.full((minSessions - 1,), np.nan)
    betweenSessImgOri  = np.full((minSessions - 1,), np.nan)
    betweenSessDist    = np.full((minSessions - 1,), np.nan)
    betweenSessDistOri = np.full((minSessions - 1,), np.nan)

    for d in range(1, minSessions):
        mask = (distMatrix == d)
        betweenSessImg[d - 1]     = _nanmean(avgImgCorrMat[mask])
        betweenSessImgOri[d - 1]  = _nanmean(avgImgCorrMatOri[mask])
        betweenSessDist[d - 1]    = _nanmean(R[mask])
        betweenSessDistOri[d - 1] = _nanmean(Ro[mask])

    # --- Permutations (deterministic; identical RNG across runs)
    rng = np.random.default_rng(1)
    if fixedFirst:
        permOrders = np.array([
            np.concatenate(([0], rng.permutation(minSessions - 1) + 1))
            for _ in range(nperms)
        ], dtype=int)  # (nperms, ns)
    else:
        permOrders = np.array([rng.permutation(minSessions) for _ in range(nperms)], dtype=int)

    imgCorrMatPerm          = np.full((nperms, minSessions, minSessions), np.nan)
    imgCorrMatOriPerm       = np.full_like(imgCorrMatPerm, np.nan)
    betweenSessCorrPerm     = np.full_like(imgCorrMatPerm, np.nan)
    betweenSessCorrOriPerm  = np.full_like(imgCorrMatPerm, np.nan)
    betweenSessImgPerm      = np.full((nperms, minSessions - 1), np.nan)
    betweenSessImgOriPerm   = np.full_like(betweenSessImgPerm, np.nan)
    betweenSessDistPerm     = np.full_like(betweenSessImgPerm, np.nan)
    betweenSessDistOriPerm  = np.full_like(betweenSessImgPerm, np.nan)

    for ip, ordv in enumerate(permOrders):
        tmp  = R[np.ix_(ordv, ordv)]
        tmpo = Ro[np.ix_(ordv, ordv)]
        imgCorrMatPerm[ip]         = tmp      # MATLAB uses betweenSessCorr for these too
        imgCorrMatOriPerm[ip]      = tmpo
        betweenSessCorrPerm[ip]    = tmp
        betweenSessCorrOriPerm[ip] = tmpo
        for d in range(1, minSessions):
            mask = (distMatrix == d)
            betweenSessImgPerm[ip, d - 1]     = _nanmean(tmp[mask])
            betweenSessImgOriPerm[ip, d - 1]  = _nanmean(tmpo[mask])
            betweenSessDistPerm[ip, d - 1]    = _nanmean(tmp[mask])
            betweenSessDistOriPerm[ip, d - 1] = _nanmean(tmpo[mask])

    # --- Pack outputs (per-subject file, like MATLAB per-subject structs)
    out = dict(
        # meta / bookkeeping
        toNormalize=0,
        toZscore=int(to_zscore),
        r2thresh=float(r2thresh),
        subjects=np.array([isub], dtype=int),
        vreg=int(vreg),
        minSessions=int(minSessions),
        distMatrix=distMatrix.astype(int),
        subSessions=subSessions,          # (ns,)
        metric=metric,

        # real metrics
        betweenSessCorr=R,                # (ns, ns)
        betweenSessCorrOri=Ro,            # (ns, ns)
        avgImgCorrMat=avgImgCorrMat,      # (ns, ns)
        avgImgCorrMatOri=avgImgCorrMatOri,
        betweenSessImg=betweenSessImg,                  # (ns-1,)
        betweenSessImgOri=betweenSessImgOri,            # (ns-1,)
        betweenSessDist=betweenSessDist,                # (ns-1,)
        betweenSessDistOri=betweenSessDistOri,          # (ns-1,)

        # permutations
        permOrders=permOrders,                              # (nperms, ns)
        imgCorrMatPerm=imgCorrMatPerm,                      # (nperms, ns, ns)
        imgCorrMatOriPerm=imgCorrMatOriPerm,
        betweenSessCorrPerm=betweenSessCorrPerm,
        betweenSessCorrOriPerm=betweenSessCorrOriPerm,
        betweenSessImgPerm=betweenSessImgPerm,              # (nperms, ns-1)
        betweenSessImgOriPerm=betweenSessImgOriPerm,
        betweenSessDistPerm=betweenSessDistPerm,
        betweenSessDistOriPerm=betweenSessDistOriPerm,

        # voxel stats
        numGoodVox=np.array(nvox_good, dtype=int)
    )

    # --- Save (per-subject)
    out_path = _out_npz_path(base_folder, vreg, isub, to_zscore, nperms, r2thresh, fixedFirst, metric)
    np.savez_compressed(out_path, **out)
    print(f"✓ Saved: {out_path}")
    return out_path

# =========================
# Batch runner (skips existing)
# =========================
def _out_exists(base_folder, vreg, isub, to_zscore, nperms, r2thresh, fixedFirst, metric):
    return os.path.exists(_out_npz_path(base_folder, vreg, isub, to_zscore, nperms, r2thresh, fixedFirst, metric))

def run_perm_pop_all(
    vreg: int = DEFAULT_VREG,
    to_zscore: int = DEFAULT_TO_ZSCORE,
    nperms: int = 1000,
    r2thresh: float = 0.0,
    fixedFirst: bool = False,
    overwrite: bool = False,
    base_folder: str = BASE_FOLDER,
    metric: str = DEFAULT_METRIC
) -> list[str]:
    """
    Auto-discovers subjects that have both inputs (regress NPZ and simPopResp NPZ),
    runs save_perm_pop_expand_npz() for each, and SKIPS those that already have outputs
    unless overwrite=True.
    """
    subs = _discover_subjects_with_inputs(base_folder, vreg, to_zscore)
    print(f"Subjects with required inputs present (v{vreg}, {_zstr(to_zscore)}): {subs}")

    out_paths = []
    for isub in subs:
        if (not overwrite) and _out_exists(base_folder, vreg, isub, to_zscore, nperms, r2thresh, fixedFirst, metric):
            print(f"• Subject {isub}: output exists ({metric}) → skip")
            continue
        try:
            p = save_perm_pop_expand_npz(
                to_zscore=to_zscore,
                nperms=nperms,
                r2thresh=r2thresh,
                fixedFirst=fixedFirst,
                isub=isub,
                vreg=vreg,
                base_folder=base_folder,
                metric=metric
            )
            out_paths.append(p)
        except Exception as e:
            print(f"[ERROR] Subject {isub} failed: {e}")

    print(f"Done. Created/updated {len(out_paths)} file(s).")
    return out_paths

# =========================
# Example usage
# =========================
# Auto-discover and run with Spearman (default), skip existing:
if __name__ == "__main__":
    run_perm_pop_all(vreg=1, to_zscore=0, nperms=1000, r2thresh=0.0, fixedFirst=False,
                     overwrite=False, metric= DEFAULT_METRIC)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Subjects with required inputs present (v1, ): [3]
✓ Saved: /content/drive/My Drive/V1_Drift/repDrift_expand/v1/Subject 3/permPopN1000_v1_sub3_rmse.npz
Done. Created/updated 1 file(s).


#check output

In [ ]:
# ==== permPop outputs shape & basic stats checker ====
import os, glob, numpy as np

BASE_FOLDER = "/content/drive/My Drive/V1_Drift/repDrift_expand"
VREG = 1
SUBJ = 3  # change if needed

def _nanpct(a):
    a = np.asarray(a)
    return 100.0 * np.isnan(a).mean()

def _symmetry_report(M, name):
    M = np.asarray(M, float)
    if M.ndim != 2:
        print(f"[{name}] not 2D, skip symmetry check");
        return
    diff = np.abs(M - M.T)
    skew = np.nanmax(diff)
    d = np.diag(M)
    print(f"{name:>22}  shape={M.shape}  nan%={_nanpct(M):5.2f}%  "
          f"max|M-M^T|={skew:.3g}  diag[min,max]=[{np.nanmin(d):.3g}, {np.nanmax(d):.3g}]")

def summarize_permPop(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    print(f"\nLoaded: {npz_path}")
    ns   = int(d["minSessions"])
    met  = str(d["metric"]) if "metric" in d.files else "unknown"
    nvox = int(np.array(d["numGoodVox"]).item()) if "numGoodVox" in d.files else -1
    print(f"minSessions={ns} | metric={met} | numGoodVox={nvox}")

    # Core session matrices
    _symmetry_report(d["betweenSessCorr"],    "betweenSessCorr")
    _symmetry_report(d["betweenSessCorrOri"], "betweenSessCorrOri")
    _symmetry_report(d["avgImgCorrMat"],      "avgImgCorrMat")
    _symmetry_report(d["avgImgCorrMatOri"],   "avgImgCorrMatOri")

    # Distance-by-lag vectors
    for nm in ["betweenSessImg","betweenSessImgOri","betweenSessDist","betweenSessDistOri"]:
        v = np.asarray(d[nm])
        print(f"{nm:>22}  shape={v.shape}  nan%={_nanpct(v):5.2f}%  "
              f"min={np.nanmin(v):.4g}  max={np.nanmax(v):.4g}")

# ==== Run checks on all files for this subject ====
subj_folder = os.path.join(BASE_FOLDER, f"v{VREG}", f"Subject {SUBJ}")
files = sorted(glob.glob(os.path.join(subj_folder, "permPopN*_v*_sub*_*.npz")))
print("Found files:")
for f in files:
    print("  ", os.path.basename(f))

for f in files:
    summarize_permPop(f)


Found files:
   permPopN1000_v1_sub3_euclidean.npz
   permPopN1000_v1_sub3_pearson.npz
   permPopN1000_v1_sub3_rmse.npz
   permPopN1000_v1_sub3_spearman.npz

Loaded: /content/drive/My Drive/V1_Drift/repDrift_expand/v1/Subject 3/permPopN1000_v1_sub3_euclidean.npz
minSessions=30 | metric=euclidean | numGoodVox=1254
       betweenSessCorr  shape=(30, 30)  nan%= 0.00%  max|M-M^T|=0  diag[min,max]=[1, 1]
    betweenSessCorrOri  shape=(30, 30)  nan%= 0.00%  max|M-M^T|=0  diag[min,max]=[1, 1]
         avgImgCorrMat  shape=(30, 30)  nan%= 0.00%  max|M-M^T|=3.33e-16  diag[min,max]=[1, 1]
      avgImgCorrMatOri  shape=(30, 30)  nan%= 0.00%  max|M-M^T|=2.22e-16  diag[min,max]=[1, 1]
        betweenSessImg  shape=(29,)  nan%= 0.00%  min=0.8205  max=0.8747
     betweenSessImgOri  shape=(29,)  nan%= 0.00%  min=0.6762  max=0.7124
       betweenSessDist  shape=(29,)  nan%= 0.00%  min=-1.351  max=-0.8356
    betweenSessDistOri  shape=(29,)  nan%= 0.00%  min=-9.884  max=-5.799

Loaded: /content/drive/My